# TP2: Audioanalyse
### Labor Versuch TP2: Abtastung, Stereo, Spektrum und Filter

In diesem Versuch lernen Sie, wie ein Computer Klang speichert, und untersuchen drei echte Aufnahmen: Sprache, Musik und das Geräusch eines Rennwagens. Am Anfang steht eine kurze Einführung in Python.

## 1 Hinweise zur Durchführung

- **Begleitheft:** Zu diesem Notebook gehört das PDF [`TP2_Audioanalyse.pdf`](https://github.com/Lamhour-Mohamed-Akram/audio-dsp-lab/blob/main/handouts/TP2_Audioanalyse.pdf). Abschnitte und Aufgaben haben im Notebook und im PDF **dieselben Nummern**.
- **Pflichtaufgaben** bearbeiten alle. **Zusatzaufgaben** sind freiwillig.
- **Antworten:** Schreiben Sie Ihre Antworten in das gedruckte Begleitheft **oder** in eine eigene Kopie dieses Notebooks. In Google Colab geht das über **Datei > Kopie in Drive speichern** (englisch *File > Save a copy in Drive*). Schreibrechte für das Original brauchen Sie nicht.
- **Zeitbedarf (Schätzung):** Python-Kurzeinführung (4.1) ca. 30 bis 45 Minuten für Einsteiger. Pflichtaufgaben 2.1 bis 2.5 danach ca. 60 bis 90 Minuten.
- **Arbeitsweise:** Führen Sie die Zellen von oben nach unten mit **Shift + Enter** aus. Ändern Sie in den Aufgaben immer nur **ein oder zwei Parameter** und führen Sie die Zelle danach erneut aus.
- **Kopfhörer:** Tiefe Töne sind über Laptop-Lautsprecher oft kaum hörbar.

**Hinweise für Google Colab**
- Eventuell müssen Sie sich mit einem Google-Konto anmelden.
- Beim ersten Ausführen kann eine Warnung erscheinen, dass das Notebook nicht von Google stammt. Bestätigen Sie mit **"Trotzdem ausführen"** (englisch *Run anyway*).
- **Sitzung neu starten** (englisch *Restart session*): Alle Variablen sind weg, die Dateien bleiben erhalten.
- **Laufzeit trennen und löschen** oder eine **neue Laufzeit**: Auch die heruntergeladenen Dateien sind weg.
- In beiden Fällen: Führen Sie die Zellen in Abschnitt 1 erneut aus und danach alle Zellen bis zu der Stelle, an der Sie arbeiten. Am einfachsten geht das mit **Laufzeit > Alle ausführen** (englisch *Runtime > Run all*).
- Eine GPU wird nicht benötigt. Dieses Notebook braucht kein anderes Notebook.

### 1.1 Einrichtung

Die nächsten zwei Zellen bereiten alles vor. Sie müssen den Code hier noch nicht verstehen.

1. Die erste Zelle prüft, ob alle benötigten Python-Pakete vorhanden sind. Nur fehlende Pakete werden installiert.
2. Die zweite Zelle stellt die Audiodateien `speech.wav`, `music.wav`, `sound.wav` im Ordner `data/` bereit. Sie lädt eine Datei **nur dann** herunter, wenn sie fehlt oder beschädigt ist.

In [ ]:
# Einrichtung 1: fehlende Pakete installieren (in Google Colab ist meist schon alles da)
import importlib.util
import subprocess
import sys

pakete = {"numpy": "numpy", "scipy": "scipy", "matplotlib": "matplotlib",
          "soundfile": "soundfile", "librosa": "librosa", "IPython": "ipython"}

fehlend = [paket for modul, paket in pakete.items() if importlib.util.find_spec(modul) is None]
if fehlend:
    print("Installiere:", ", ".join(fehlend))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *fehlend])
    print("Fertig.")
else:
    print("Alle Pakete sind bereits installiert.")

In [ ]:
# Einrichtung 2: Audiodateien bereitstellen (Download nur, wenn eine Datei fehlt oder beschädigt ist)
import os
import urllib.request
import soundfile as sf

DATEN_URL = "https://raw.githubusercontent.com/Lamhour-Mohamed-Akram/audio-dsp-lab/afe9e3d4fc8266c132421123ecd92235bc046779/data/"
ERWARTET = {"speech.wav": (16000, 1), "music.wav": (44100, 2), "sound.wav": (22050, 2)}   # Datei: (Abtastrate, Kanäle)


def audiodatei_ok(pfad, abtastrate, kanaele):
    """Prüft, ob die Datei eine lesbare Audiodatei im erwarteten Format ist."""
    try:
        info = sf.info(pfad)
    except Exception:
        return False
    return info.samplerate == abtastrate and info.channels == kanaele and info.frames > 0


os.makedirs("data", exist_ok=True)
for name, (abtastrate, kanaele) in ERWARTET.items():
    pfad = os.path.join("data", name)
    if audiodatei_ok(pfad, abtastrate, kanaele):
        print(name, ": vorhanden")
        continue
    print(name, ": wird heruntergeladen ...")
    try:
        with urllib.request.urlopen(DATEN_URL + name, timeout=60) as antwort:
            inhalt = antwort.read()
    except Exception as fehler:
        raise RuntimeError("Download von " + name + " fehlgeschlagen. Bitte Internetverbindung prüfen "
                           "und diese Zelle erneut ausführen. Grund: " + str(fehler))
    with open(pfad + ".tmp", "wb") as datei:
        datei.write(inhalt)
    if not audiodatei_ok(pfad + ".tmp", abtastrate, kanaele):
        os.remove(pfad + ".tmp")
        raise RuntimeError(name + " wurde geladen, ist aber keine gültige Audiodatei im erwarteten Format.")
    os.replace(pfad + ".tmp", pfad)
    print(name, ": OK")

print("Alle Audiodateien sind bereit.")

## 2 Lernziele

Nach diesem Versuch können Sie ...
- einfache Python-Befehle in einem Notebook ausführen und leicht verändern,
- eine Audiodatei laden und **Abtastrate, Samples, Dauer, Amplitude und Kanäle** bestimmen,
- Aufnahmen **anhören**, als **Wellenform** darstellen und einen **Ausschnitt** vergrößern,
- **Stereo** und **Mono** unterscheiden,
- mit der **FFT** ein **Frequenzspektrum** berechnen und vorsichtig interpretieren,
- einen **Tiefpass** und einen **Hochpass** anwenden und die Wirkung hören und sehen.

## 3 Vorbereitungsaufgaben

Bearbeiten Sie diese Fragen **vor** dem Labor. Kurze Antworten genügen.

- **V1:** Eine Aufnahme hat die Abtastrate 16000 Hz und dauert 3 Sekunden. Wie viele Samples hat sie?
- **V2:** Die höchste Frequenz, die ein digitales Signal enthalten kann, ist die halbe Abtastrate (**Nyquist-Frequenz**). Wie hoch ist sie bei 16000 Hz und bei 44100 Hz?
- **V3:** Was ist der Unterschied zwischen einer **Mono**-Aufnahme und einer **Stereo**-Aufnahme?
- **V4:** Was macht ein **Tiefpass**? Was macht ein **Hochpass**?

**Ihre Antworten:**
- V1: 
- V2: 
- V3: 
- V4: 

## 4 Durchführung

### 4.1 Python-Kurzeinführung (Pflicht, Übungen P1 bis P5)

Hier lernen Sie nur das, was Sie für die Labore brauchen. Wer schon Python kann, arbeitet diesen Teil zügig durch.

#### Text-Zellen und Code-Zellen

Ein Notebook besteht aus **Zellen**:
- **Text-Zellen** (wie diese) enthalten Erklärungen.
- **Code-Zellen** enthalten Python-Code. Klicken Sie in die Zelle und drücken Sie **Shift + Enter**. Das Ergebnis erscheint direkt unter der Zelle.

Alles hinter `#` ist ein **Kommentar**. Python ignoriert Kommentare. Mit `print(...)` geben Sie etwas aus.

In [ ]:
# Das ist ein Kommentar
print("Hallo! Meine erste Code-Zelle läuft.")

#### Variablen, Rechnen und Reihenfolge

Eine **Variable** ist ein Name für einen Wert. Mit `=` speichern Sie einen Wert unter diesem Namen. Python rechnet mit `+`, `-`, `*`, `/` und `**` (hoch).

**Die Reihenfolge zählt.** Eine Variable existiert erst, wenn die Zelle ausgeführt wurde, in der sie festgelegt wird. Wenn Sie eine Zelle überspringen, erscheint später ein Fehler wie `NameError`.

In [ ]:
abtastrate = 16000      # Werte pro Sekunde (Hz)
dauer = 2.5             # Sekunden (Kommazahlen schreibt man mit Punkt)

anzahl_samples = abtastrate * dauer
print("Anzahl der Werte:", anzahl_samples)

In [ ]:
# Diese Zelle benutzt die Variable "dauer" aus der Zelle darüber
print("Dauer in Millisekunden:", dauer * 1000)
print("2 hoch 10 =", 2 ** 10)

> **Übung P1:** Ändern Sie `dauer` auf `10` und führen Sie **beide** Zellen erneut aus. Wie viele Werte hat eine 10 Sekunden lange Aufnahme mit 16000 Hz?

#### Listen, Index ab 0 und Ausschnitte

Eine **Liste** speichert mehrere Werte in eckigen Klammern. Mit einem **Index** holen Sie einen einzelnen Wert heraus.
- Python beginnt beim Zählen mit **0**. Das erste Element ist `werte[0]`.
- `werte[-1]` ist das letzte Element.
- Ein **Ausschnitt** (englisch *slicing*) `werte[1:4]` enthält die Elemente mit Index 1, 2 und 3. Das Ende (4) gehört **nicht** dazu.

In [ ]:
werte = [0.0, 0.5, 1.0, 0.5, 0.0, -0.5, -1.0]

print("Ganze Liste:     ", werte)
print("Erstes Element:  ", werte[0])
print("Drittes Element: ", werte[2])
print("Letztes Element: ", werte[-1])
print("Ausschnitt [1:4]:", werte[1:4])
print("Anzahl Elemente: ", len(werte))

> **Übung P2:** Ändern Sie die Zahlen in der eckigen Klammer so, dass genau die **ersten drei** Elemente ausgegeben werden.

In [ ]:
print(werte[0:1])    # ändern Sie die Zahlen

#### `import` und NumPy-Arrays

Mit `import` laden Sie eine **Bibliothek**. Für Zahlenreihen benutzen wir **NumPy** mit dem Kurznamen `np`. Ein **NumPy-Array** ist wie eine Liste, aber man kann direkt damit rechnen: `x * 2` verdoppelt **jeden** Wert. Ein Audiosignal ist in Python genau so ein Array.

In [ ]:
import numpy as np

x = np.array(werte)          # Liste in ein NumPy-Array umwandeln

print("x      =", x)
print("x * 2  =", x * 2)                       # jeder Wert mal 2
print("x[2:5] =", x[2:5])                      # Ausschnitt
print("Maximum:", np.max(x))
print("Größter Betrag:", np.max(np.abs(x)))    # Betrag = Wert ohne Vorzeichen

#### `for`-Schleifen und `if`-Abfragen

- Eine **`for`-Schleife** wiederholt Code für jedes Element.
- Eine **`if`-Abfrage** führt Code nur aus, wenn eine Bedingung wahr ist. `else` gilt für alle anderen Fälle.
- Der Code **innerhalb** von `for` und `if` ist **eingerückt** (4 Leerzeichen).

Audio-Samples können positiv oder negativ sein. Für "laut oder leise" zählt der **Betrag** `abs(wert)`: Der Wert -1.0 ist genauso weit von 0 entfernt wie +1.0.

In [ ]:
schwelle = 0.4

for wert in werte:
    if abs(wert) > schwelle:          # abs() = Betrag, das Vorzeichen spielt keine Rolle
        print(wert, "ist weit von 0 entfernt (groß)")
    else:
        print(wert, "ist nah bei 0 (klein)")

Oft zählt man mit `range(...)`. Mit `append` hängt man einen Wert hinten an eine Liste an.

In [ ]:
quadrate = []                  # eine leere Liste
for i in range(4):             # range(4) zählt 0, 1, 2, 3
    quadrate.append(i ** 2)    # Quadrat hinten anhängen
print(quadrate)

> **Übung P3:** Setzen Sie oben `schwelle = 0.8` und führen Sie die Zelle erneut aus. Welche Werte gelten jetzt als "groß"?
>
> *Hinweis:* Ein einzelnes Sample sagt wenig darüber, ob gerade gesprochen wird. Die Sprachaktivitätserkennung in TP4 benutzt deshalb die **Energie in kurzen Zeitfenstern** (viele Samples zusammen).

#### Funktionen aufrufen und selbst schreiben

Eine **Funktion** ist ein Baustein, den man mit Namen und runden Klammern aufruft: `name(eingabe)`. Mehrere Eingaben trennt man mit Komma. Manche Eingaben haben einen Namen, zum Beispiel `rate=16000`.

Mit `def` schreiben Sie eine eigene Funktion. `return` gibt das Ergebnis zurück. Der Text in dreifachen Anführungszeichen beschreibt, was die Funktion tut.

In [ ]:
print(round(3.14159, 2))     # runden auf 2 Nachkommastellen
print(np.mean(x))            # Mittelwert des Arrays x

In [ ]:
def sekunden_zu_samples(sekunden, abtastrate):
    """Rechnet eine Zeit in Sekunden in eine Anzahl von Samples um."""
    anzahl = int(round(sekunden * abtastrate))    # round() rundet, int() macht eine ganze Zahl daraus
    return anzahl

print(sekunden_zu_samples(2.0, 16000))
print(sekunden_zu_samples(0.5, 44100))

> **Übung P4:** Wie viele Samples sind **25 Millisekunden** bei 16000 Hz? Ändern Sie die Zahl im Aufruf. (25 ms = 0.025 s)

In [ ]:
print(sekunden_zu_samples(0.0, 16000))    # ändern Sie 0.0

#### Kurz erklärt: Ein Ton ist eine Zahlenfolge

Ein Computer speichert Klang als **Folge von Zahlen** (Samples). Die **Abtastrate** sagt, wie viele Zahlen pro Sekunde gespeichert werden. Wir erzeugen einen Sinuston mit 440 Hz und zeichnen die ersten 10 Millisekunden mit der Bibliothek **matplotlib** (Kurzname `plt`).

*Synthetische Töne dienen nur zur Erklärung. Die Aufgaben arbeiten mit echten Aufnahmen.*

In [ ]:
import matplotlib.pyplot as plt

sr_ton = 16000                                # Abtastrate in Hz
t = np.arange(sr_ton) / sr_ton                # Zeitachse: 1 Sekunde
ton = 0.3 * np.sin(2 * np.pi * 440 * t)       # Sinuston: 440 Hz, Amplitude 0.3

plt.figure(figsize=(10, 3))
plt.plot(t[:160] * 1000, ton[:160], marker=".")    # 160 Samples = 10 ms
plt.xlabel("Zeit [ms]")
plt.ylabel("Amplitude")
plt.title("Sinuston 440 Hz: jeder Punkt ist ein Sample")
plt.grid(True)
plt.show()

#### Eine echte Aufnahme öffnen, zeichnen und anhören

- `sf.read(...)` aus der Bibliothek **soundfile** liest eine WAV-Datei und gibt **zwei** Dinge zurück: das Signal und die Abtastrate.
- Die Funktion `anhoeren(...)` zeigt einen Abspieler. Sie verändert die Lautstärke **nicht** automatisch. So hören Sie echte Lautstärkeunterschiede. Mit `skala` kann man ein Signal leiser machen (mehr dazu in 4.6).

In [ ]:
from IPython.display import Audio, display

def anhoeren(signal, sr, text="", skala=1.0):
    """Zeigt einen Audio-Abspieler, ohne automatische Lautstärke-Anpassung."""
    signal = np.asarray(signal) * skala
    if np.max(np.abs(signal)) > 1:
        print("Achtung: Werte über 1 werden für die Wiedergabe abgeschnitten. Bitte kleinere Skala wählen.")
        signal = np.clip(signal, -1, 1)
    if signal.ndim == 2:        # Stereo: der Abspieler erwartet (Kanäle, Samples)
        signal = signal.T
    if text:
        print(text)
    display(Audio(signal, rate=sr, normalize=False))

In [ ]:
import soundfile as sf

sprache, sr_sprache = sf.read("data/speech.wav")

zeit = np.arange(len(sprache)) / sr_sprache     # Zeitachse in Sekunden

plt.figure(figsize=(10, 3))
plt.plot(zeit, sprache, linewidth=0.5)
plt.xlabel("Zeit [s]")
plt.ylabel("Amplitude")
plt.title("Sprachaufnahme speech.wav")
plt.grid(True)
plt.show()

anhoeren(sprache, sr_sprache, "Sprache:")

> **Übung P5:** Hören Sie die Aufnahme an. Wo im Diagramm sehen Sie **Pausen**? Nennen Sie zwei ungefähre Zeitpunkte.

**Ihre Antworten zu P1 bis P5:** 

### 4.2 Aufnahmen laden und beschreiben (Aufgabe 2.1, Pflicht)

Jede WAV-Datei hat einen "Steckbrief":
- **Abtastrate** (Hz): Samples pro Sekunde
- **Kanäle**: 1 = Mono, 2 = Stereo
- **Samples**: Anzahl der gespeicherten Werte pro Kanal
- **Dauer** = Samples geteilt durch Abtastrate
- **Format**: `PCM_16` bedeutet 16-Bit-Ganzzahlen. `sf.read` rechnet sie in Kommazahlen zwischen -1 und +1 um.
- **Spitzenwert**: größter Betrag im Signal.
- **Werte nahe Vollaussteuerung**: Anzahl der Samples mit Betrag ab 0.999. Das ist nur ein **Hinweis** auf mögliche Übersteuerung (englisch *clipping*), **kein Beweis**. Ein Spitzenwert nahe 1 kann auch ohne Übersteuerung vorkommen. Echte Übersteuerung erkennt man zum Beispiel an abgeflachten Spitzen in der Wellenform.

In [ ]:
def steckbrief(dateiname):
    """Liest eine Audiodatei und zeigt ihre wichtigsten Eigenschaften."""
    info = sf.info("data/" + dateiname)
    signal, sr = sf.read("data/" + dateiname)
    print("Datei:        ", dateiname)
    print("Abtastrate:   ", sr, "Hz")
    print("Kanäle:       ", info.channels)
    print("Samples:      ", len(signal))
    print("Dauer:        ", round(len(signal) / sr, 2), "s")
    print("Format:       ", info.subtype)
    print("Spitzenwert:  ", round(np.max(np.abs(signal)), 3))
    print("Samples mit Betrag ab 0.999 (Hinweis):", np.sum(np.abs(signal) >= 0.999))
    print()

for name in ["speech.wav", "music.wav", "sound.wav"]:
    steckbrief(name)

Jetzt laden wir alle drei Aufnahmen in Variablen. `.shape` zeigt die **Form** eines Arrays:
- `(Samples,)` bei Mono: eine lange Zahlenreihe
- `(Samples, 2)` bei Stereo: eine Tabelle mit zwei Spalten, links und rechts

In [ ]:
sprache, sr_sprache = sf.read("data/speech.wav")
musik, sr_musik = sf.read("data/music.wav")
geraeusch, sr_geraeusch = sf.read("data/sound.wav")

print("Sprache: ", sprache.shape, "bei", sr_sprache, "Hz")
print("Musik:   ", musik.shape, "bei", sr_musik, "Hz")
print("Geräusch:", geraeusch.shape, "bei", sr_geraeusch, "Hz")

> **Aufgabe 2.1:** Tragen Sie die Werte in **Tabelle A** (Abschnitt 5) ein. Welche Datei hat die höchste Abtastrate? Welche Dateien sind Stereo? Gibt es Hinweise auf Übersteuerung?

**Ihre Antwort:** 

### 4.3 Wellenform, Zeitpunkt und Ausschnitt (Aufgabe 2.2, Pflicht)

Die **Wellenform** zeigt die Amplitude über der Zeit. Bei Stereo zeichnen wir beide Kanäle übereinander. `signal[:, 0]` bedeutet: alle Zeilen (`:`) aus Spalte 0, also der linke Kanal. `signal[:, 1]` ist der rechte Kanal.

In [ ]:
def wellenform(signal, sr, titel):
    """Zeichnet die Wellenform. Bei Stereo werden beide Kanäle gezeichnet."""
    zeit = np.arange(len(signal)) / sr
    plt.figure(figsize=(10, 2.5))
    if signal.ndim == 1:
        plt.plot(zeit, signal, linewidth=0.5)
    else:
        plt.plot(zeit, signal[:, 0], linewidth=0.5, label="links")
        plt.plot(zeit, signal[:, 1], linewidth=0.5, label="rechts", alpha=0.7)
        plt.legend(loc="upper right")
    plt.ylim(-1, 1)
    plt.xlabel("Zeit [s]")
    plt.ylabel("Amplitude")
    plt.title(titel)
    plt.grid(True)
    plt.show()

In [ ]:
wellenform(sprache, sr_sprache, "Sprache (mono)")
anhoeren(sprache, sr_sprache, "Sprache:")

wellenform(musik, sr_musik, "Musik (stereo)")
anhoeren(musik, sr_musik, "Musik:")

wellenform(geraeusch, sr_geraeusch, "Geräusch: Rennwagen (stereo)")
anhoeren(geraeusch, sr_geraeusch, "Geräusch:")

**Zeit und Index.** Um eine Zeit in einen Index umzurechnen, multipliziert man mit der Abtastrate **der jeweiligen Aufnahme**. Bei der Musik brauchen wir also das Array `musik` **und** die Abtastrate `sr_musik`. Bei Stereo liefert ein Index zwei Werte: links und rechts.

In [ ]:
zeitpunkt_s = 1.5          # Zeitpunkt in Sekunden

index_sprache = sekunden_zu_samples(zeitpunkt_s, sr_sprache)
index_musik = sekunden_zu_samples(zeitpunkt_s, sr_musik)

print("Sprache: Index", index_sprache, "  Wert:", round(sprache[index_sprache], 4))
print("Musik:   Index", index_musik, "  Werte (links, rechts):", np.round(musik[index_musik], 4))

**Ausschnitt.** Die Funktion `ausschnitt` schneidet einen Teil heraus. Sie prüft, ob der Bereich innerhalb der Aufnahme liegt, und meldet sonst einen verständlichen Fehler. Gültig: `start_s` ab 0, `laenge_s` größer als 0, und `start_s + laenge_s` höchstens so groß wie die Dauer.

In [ ]:
def ausschnitt(signal, sr, start_s, laenge_s):
    """Schneidet einen Abschnitt heraus und prüft, ob er in der Aufnahme liegt."""
    dauer_s = len(signal) / sr
    if laenge_s <= 0:
        raise ValueError("laenge_s muss größer als 0 sein.")
    if start_s < 0 or start_s + laenge_s > dauer_s + 1e-9:
        raise ValueError("Der Ausschnitt von " + str(start_s) + " s bis " + str(start_s + laenge_s)
                         + " s liegt nicht vollständig in der Aufnahme (Dauer " + str(round(dauer_s, 2)) + " s).")
    start = sekunden_zu_samples(start_s, sr)
    ende = min(start + sekunden_zu_samples(laenge_s, sr), len(signal))
    if ende <= start:
        raise ValueError("Der Ausschnitt ist kürzer als ein Sample.")
    return signal[start:ende]

In [ ]:
start_s = 4.0       # Beginn des Ausschnitts in Sekunden (hier liegt ein Vokal)
laenge_s = 0.05     # Länge in Sekunden (0.05 s = 50 ms)

stueck = ausschnitt(sprache, sr_sprache, start_s, laenge_s)
zeit_ms = start_s * 1000 + np.arange(len(stueck)) / sr_sprache * 1000

plt.figure(figsize=(10, 3))
plt.plot(zeit_ms, stueck)
plt.xlabel("Zeit [ms]")
plt.ylabel("Amplitude")
plt.title("Ausschnitt aus der Sprachaufnahme")
plt.grid(True)
plt.show()

hoer_start_s = 3.0      # längerer Abschnitt zum Anhören: Beginn in Sekunden
hoer_laenge_s = 3.0     # Länge in Sekunden
anhoeren(ausschnitt(sprache, sr_sprache, hoer_start_s, hoer_laenge_s), sr_sprache, "Ausgewählter Abschnitt:")

> **Aufgabe 2.2:**
> a) Beschreiben Sie in **Tabelle B** kurz die drei Wellenformen und Ihren Höreindruck.
> b) Warum gehören bei `zeitpunkt_s = 1.5` zu Sprache und Musik verschiedene Indizes?
> c) Setzen Sie `start_s` auf einen Zeitpunkt in einer **Pause** (Übung P5). Wie sieht der Ausschnitt jetzt aus? Achten Sie auf die y-Achse.
> d) Setzen Sie wieder `start_s = 4.0` und dann `laenge_s = 0.5`. Was sehen Sie jetzt von den einzelnen Schwingungen?

**Ihre Antworten:**
- b) 
- c) 
- d) 

### 4.4 Stereo und Mono (Aufgabe 2.3, Pflicht)

Eine **Stereo**-Aufnahme hat zwei Kanäle. Mit Kopfhörern kommt der linke Kanal links und der rechte Kanal rechts an. Dadurch hören wir, wo ein Instrument steht.

- **Mono-Mischung:** (links + rechts) / 2. Aus beiden Seiten kommt dasselbe Signal.
- **Differenz:** (links - rechts) / 2. Sie enthält, was sich zwischen den Kanälen unterscheidet. Klänge, die in beiden Kanälen gleich sind (zum Beispiel genau in der Mitte), heben sich darin auf.
- **RMS-Wert** (Effektivwert): Wurzel aus dem Mittelwert der quadrierten Samples. Er beschreibt den **Signalpegel** und ist nur ein **grobes Maß** für die empfundene Lautstärke. Das Gehör bewertet tiefe, mittlere und hohe Frequenzen unterschiedlich.

Wir benutzen die ersten 8 Sekunden der Musik.

In [ ]:
musik_kurz = ausschnitt(musik, sr_musik, 0.0, 8.0)    # erste 8 Sekunden, beide Kanäle

links = musik_kurz[:, 0]
rechts = musik_kurz[:, 1]
mono = (links + rechts) / 2
differenz = (links - rechts) / 2

print("RMS links:    ", round(np.sqrt(np.mean(links**2)), 4))
print("RMS rechts:   ", round(np.sqrt(np.mean(rechts**2)), 4))
print("RMS Differenz:", round(np.sqrt(np.mean(differenz**2)), 4))

In [ ]:
nur_links = musik_kurz.copy()
nur_links[:, 1] = 0          # rechten Kanal stumm schalten

nur_rechts = musik_kurz.copy()
nur_rechts[:, 0] = 0         # linken Kanal stumm schalten

anhoeren(musik_kurz, sr_musik, "Original (stereo):")
anhoeren(nur_links, sr_musik, "Nur linker Kanal:")
anhoeren(nur_rechts, sr_musik, "Nur rechter Kanal:")
anhoeren(mono, sr_musik, "Mono-Mischung (links + rechts) / 2:")
anhoeren(differenz, sr_musik, "Differenz (links - rechts) / 2:")

> **Aufgabe 2.3:** Hören Sie mit Kopfhörern.
> a) Welche Instrumente hören Sie eher links, welche eher rechts?
> b) Was hören Sie in der Differenz? Was fehlt dort im Vergleich zum Original?
> c) Beschreiben Sie den Unterschied zwischen dem Original (stereo) und der Mono-Mischung.

**Ihre Antworten:**
- a) 
- b) 
- c) 

#### Zusatzaufgabe Z2.1: Korrelation der Kanäle (optional)

Die **Korrelation** zeigt, wie stark zwei Signale **linear** zusammenhängen. Sie liegt zwischen -1 und +1:
- nahe **+1**: Beide Kanäle verlaufen fast gleich (einer ist ungefähr ein Vielfaches des anderen).
- nahe **-1**: Die Kanäle verlaufen gegenläufig (einer ist ungefähr das umgekehrte Vielfache des anderen).
- nahe **0**: Es gibt **keinen linearen** Zusammenhang. Die Kanäle können trotzdem ähnliche Klänge enthalten, zum Beispiel zeitlich verschoben.

In [ ]:
print("Korrelation links/rechts, Musik:    ", round(np.corrcoef(musik[:, 0], musik[:, 1])[0, 1], 3))
print("Korrelation links/rechts, Rennwagen:", round(np.corrcoef(geraeusch[:, 0], geraeusch[:, 1])[0, 1], 3))

> **Z2.1:** Vergleichen Sie die beiden Werte. Welche Aufnahme hat den stärkeren linearen Zusammenhang zwischen links und rechts? Passt das zu Ihrem Höreindruck?

### 4.5 Frequenzspektrum mit der FFT (Aufgabe 2.4, Pflicht)

Die Wellenform zeigt das Signal über der **Zeit**. Das **Spektrum** zeigt, **welche Frequenzen** im Signal enthalten sind. Die **FFT** (Fast Fourier Transform) ist ein schnelles Rechenverfahren dafür.

- x-Achse: Frequenz in Hz
- y-Achse: Stärke in **dB** (Dezibel). Ein Sinus mit Amplitude 1 ergibt 0 dB. **-20 dB** bedeutet 10-mal kleinere Amplitude, **-40 dB** 100-mal kleiner.

Die Funktion `spektrum` berechnet ein **einseitiges Amplitudenspektrum**: Für Frequenzen zwischen 0 Hz und der halben Abtastrate wird der Anteil der negativen Frequenzen mitgezählt (Faktor 2). Der Gleichanteil bei 0 Hz und, bei gerader Länge, die Nyquist-Frequenz kommen nur einmal vor und werden deshalb nicht verdoppelt. Stereo wird für die Analyse zu Mono gemittelt.

In [ ]:
def spektrum(signal, sr, glaettung_hz=0):
    """Einseitiges Amplitudenspektrum in dB. Ein Sinus mit Amplitude 1 ergibt 0 dB."""
    if signal.ndim == 2:
        signal = signal.mean(axis=1)                 # Stereo zu Mono (nur für die Analyse)
    n = len(signal)
    if n < 2:
        raise ValueError("Das Signal ist zu kurz für eine FFT (mindestens 2 Samples).")
    betrag = np.abs(np.fft.rfft(signal)) / n
    betrag[1:] = 2 * betrag[1:]                      # positive Frequenzen: Anteil der negativen dazuzählen
    if n % 2 == 0:
        betrag[-1] = betrag[-1] / 2                  # die Nyquist-Frequenz gibt es nur einmal
    frequenzen = np.fft.rfftfreq(n, d=1 / sr)
    breite = int(glaettung_hz * n / sr)              # Glättung über glaettung_hz Hertz (0 = aus)
    if breite > 1:
        betrag = np.convolve(betrag, np.ones(breite) / breite, mode="same")
    return frequenzen, 20 * np.log10(betrag + 1e-12)

**Kurz erklärt mit einem Testton:** Zwei Sinustöne mit 440 Hz (Amplitude 0.3) und 1000 Hz (Amplitude 0.1) ergeben zwei Spitzen bei etwa -10.5 dB und -20 dB. Denn 20 mal log10(0.3) ist etwa -10.5.

In [ ]:
testton = 0.3 * np.sin(2 * np.pi * 440 * t) + 0.1 * np.sin(2 * np.pi * 1000 * t)   # t und sr_ton aus 4.1
f_test, db_test = spektrum(testton, sr_ton)

plt.figure(figsize=(10, 3))
plt.plot(f_test, db_test)
plt.xlim(0, 2000)
plt.ylim(-80, 5)
plt.xlabel("Frequenz [Hz]")
plt.ylabel("Stärke [dB]")
plt.title("Spektrum des Testtons: zwei Spitzen")
plt.grid(True)
plt.show()

**Jetzt echte Aufnahmen.** Echte Sprache und Musik bestehen **nicht** aus wenigen festen Tönen. Ihre Frequenzen **ändern sich ständig**: andere Laute, andere Noten, andere Instrumente. Die FFT über die **ganze Datei** mischt alle Momente zusammen. Deshalb sieht man breite "Hügel" statt weniger scharfer Spitzen.

Zwei Einstellungen verändern, was man ablesen kann:
- **Glättung** (`glaettung_hz=20`): Die Kurve wird über 20 Hz gemittelt und damit lesbar. Einzelne scharfe Spitzen werden dabei **niedriger und breiter**.
- **Normierung auf das eigene Maximum**: Jede Kurve wird so verschoben, dass ihr Maximum bei 0 dB liegt. Man kann dann nur die **Form** vergleichen (wo liegt relativ viel Energie?), aber **nicht**, welche Aufnahme insgesamt mehr Energie hat.

In [ ]:
max_frequenz = 8000     # bis zu welcher Frequenz gezeigt wird (Hz), gültig: bis 22050

f1, db1 = spektrum(sprache, sr_sprache, glaettung_hz=20)
f2, db2 = spektrum(musik, sr_musik, glaettung_hz=20)
f3, db3 = spektrum(geraeusch, sr_geraeusch, glaettung_hz=20)

plt.figure(figsize=(10, 4))
plt.plot(f1, db1 - np.max(db1), label="Sprache", linewidth=0.8)      # jede Kurve: eigenes Maximum = 0 dB
plt.plot(f2, db2 - np.max(db2), label="Musik", linewidth=0.8)
plt.plot(f3, db3 - np.max(db3), label="Rennwagen", linewidth=0.8)
plt.xlim(0, max_frequenz)
plt.ylim(-80, 5)
plt.xlabel("Frequenz [Hz]")
plt.ylabel("Stärke relativ zum eigenen Maximum [dB]")
plt.title("Geglättete Spektren der ganzen Dateien (nur die Form ist vergleichbar)")
plt.legend()
plt.grid(True)
plt.show()

Nimmt man nur einen **kurzen** Ausschnitt eines Vokals (100 ms), ändert sich der Klang in dieser Zeit kaum. Dann sieht man auch bei Sprache Spitzen: die **Grundfrequenz** der Stimme und ihre Vielfachen (**Obertöne**). Hier ohne Glättung.

In [ ]:
fft_start_s = 4.0       # Beginn des Ausschnitts in Sekunden
fft_laenge_s = 0.1      # Länge in Sekunden

f_kurz, db_kurz = spektrum(ausschnitt(sprache, sr_sprache, fft_start_s, fft_laenge_s), sr_sprache)

plt.figure(figsize=(10, 3))
plt.plot(f_kurz, db_kurz)
plt.xlim(0, 2000)
plt.ylim(np.max(db_kurz) - 60, np.max(db_kurz) + 5)     # zeigt die oberen 60 dB
plt.xlabel("Frequenz [Hz]")
plt.ylabel("Stärke [dB]")
plt.title("Spektrum eines kurzen Ausschnitts (100 ms)")
plt.grid(True)
plt.show()

> **Aufgabe 2.4:**
> a) Lesen Sie aus den geglätteten Spektren ab: In welchem Frequenzbereich liegt bei der **Sprache** relativ zu ihrem Maximum die meiste Energie?
> b) Welche Kurve fällt zu hohen Frequenzen hin am langsamsten ab? Warum darf man daraus nicht schließen, dass diese Aufnahme insgesamt lauter ist?
> c) Schätzen Sie im kurzen Ausschnitt den Abstand zwischen zwei benachbarten Spitzen. Ändern Sie danach `fft_start_s` auf `5.0` und auf `2.5`. Was beobachten Sie?

**Ihre Antworten:**
- a) 
- b) 
- c) 

### 4.6 Tiefpass und Hochpass (Aufgabe 2.5, Pflicht)

Ein **Filter** verändert, welche Frequenzen durchkommen.

| Filter | lässt durch | dämpft |
|---|---|---|
| Tiefpass | tiefe Frequenzen | hohe Frequenzen |
| Hochpass | hohe Frequenzen | tiefe Frequenzen |

Die **Grenzfrequenz** legt fest, wo der Filter anfängt zu dämpfen. Gültig sind Werte größer als 0 und kleiner als die halbe Abtastrate, sonst meldet die Funktion einen Fehler. Wir benutzen einen fertigen **Butterworth**-Filter aus der Bibliothek **scipy**. Die **Ordnung** bestimmt, wie steil er ist.

- `sosfiltfilt` wendet den Filter **vorwärts und rückwärts** an. Dadurch wird das Signal im Inneren **nicht zeitlich verschoben** (Nullphase).
- Die Dämpfung wirkt dabei **zweimal**: Der Betrag des Frequenzgangs wird quadriert. Bei der Grenzfrequenz ist das Signal deshalb um etwa 6 dB gedämpft statt um 3 dB.
- Ganz am Anfang und am Ende des Signals können kleine **Randeffekte** auftreten.
- `axis=0` sorgt dafür, dass bei Stereo jeder Kanal einzeln gefiltert wird. Das Stereobild bleibt erhalten.

In [ ]:
from scipy import signal as sig

def pruefe_grenzfrequenz(grenzfrequenz, sr):
    """Gültig sind Grenzfrequenzen größer als 0 und kleiner als die halbe Abtastrate."""
    if not 0 < grenzfrequenz < sr / 2:
        raise ValueError("Grenzfrequenz " + str(grenzfrequenz) + " Hz ist ungültig. Erlaubt: größer als 0 "
                         "und kleiner als " + str(sr / 2) + " Hz (halbe Abtastrate).")

def tiefpass(x, sr, grenzfrequenz, ordnung=4):
    """Butterworth-Tiefpass, vorwärts und rückwärts angewendet."""
    pruefe_grenzfrequenz(grenzfrequenz, sr)
    sos = sig.butter(ordnung, grenzfrequenz, btype="lowpass", fs=sr, output="sos")
    return sig.sosfiltfilt(sos, x, axis=0)

def hochpass(x, sr, grenzfrequenz, ordnung=4):
    """Butterworth-Hochpass, vorwärts und rückwärts angewendet."""
    pruefe_grenzfrequenz(grenzfrequenz, sr)
    sos = sig.butter(ordnung, grenzfrequenz, btype="highpass", fs=sr, output="sos")
    return sig.sosfiltfilt(sos, x, axis=0)

**Faire Hörvergleiche:** Original und gefiltertes Signal werden mit **demselben Faktor** `skala` abgespielt. Nur wenn ein Signal sonst über 1 hinausgehen würde, werden **alle** Signale gleich stark leiser gemacht. So bleiben echte Lautstärkeunterschiede hörbar.

In [ ]:
def gemeinsame_skala(*signale):
    """Ein gemeinsamer Faktor für alle Signale. Er ist nur kleiner als 1, wenn sonst etwas übersteuern würde."""
    spitze = max(np.max(np.abs(s)) for s in signale)
    if spitze == 0:
        return 1.0
    return min(1.0, 0.99 / spitze)

**Tiefpass auf Musik**

In [ ]:
grenzfrequenz_tp = 1000      # Hz, Vorschläge: 300, 1000, 4000 (gültig: größer als 0, kleiner als 22050)

musik_tp = tiefpass(musik_kurz, sr_musik, grenzfrequenz_tp)

skala = gemeinsame_skala(musik_kurz, musik_tp)
print("Gemeinsame Skala:", round(skala, 3))
anhoeren(musik_kurz, sr_musik, "Original:", skala)
anhoeren(musik_tp, sr_musik, "Tiefpass:", skala)

f_orig, db_orig = spektrum(musik_kurz, sr_musik, glaettung_hz=20)
f_tp, db_tp = spektrum(musik_tp, sr_musik, glaettung_hz=20)
plt.figure(figsize=(10, 3))
plt.plot(f_orig, db_orig, label="Original", linewidth=0.8)
plt.plot(f_tp, db_tp, label="Tiefpass", linewidth=0.8)     # gleiche Skala: Kurven direkt vergleichbar
plt.axvline(grenzfrequenz_tp, color="red", linestyle="--", label="Grenzfrequenz")
plt.xlim(0, 10000)
plt.ylim(np.max(db_orig) - 80, np.max(db_orig) + 5)
plt.xlabel("Frequenz [Hz]")
plt.ylabel("Stärke [dB]")
plt.title("Musik: geglättetes Spektrum vor und nach dem Tiefpass")
plt.legend()
plt.grid(True)
plt.show()

**Hochpass auf Sprache**

In [ ]:
grenzfrequenz_hp = 500     # Hz, Vorschläge: 100, 500, 2000 (gültig: größer als 0, kleiner als 8000)

sprache_hp = hochpass(sprache, sr_sprache, grenzfrequenz_hp)

skala = gemeinsame_skala(sprache, sprache_hp)
anhoeren(sprache, sr_sprache, "Original:", skala)
anhoeren(sprache_hp, sr_sprache, "Hochpass:", skala)
print("RMS Original:", round(np.sqrt(np.mean(sprache**2)), 4), "  RMS Hochpass:", round(np.sqrt(np.mean(sprache_hp**2)), 4))

f_orig, db_orig = spektrum(sprache, sr_sprache, glaettung_hz=20)
f_hp, db_hp = spektrum(sprache_hp, sr_sprache, glaettung_hz=20)
plt.figure(figsize=(10, 3))
plt.plot(f_orig, db_orig, label="Original", linewidth=0.8)
plt.plot(f_hp, db_hp, label="Hochpass", linewidth=0.8)
plt.axvline(grenzfrequenz_hp, color="red", linestyle="--", label="Grenzfrequenz")
plt.xlim(0, 4000)
plt.ylim(np.max(db_orig) - 80, np.max(db_orig) + 5)
plt.xlabel("Frequenz [Hz]")
plt.ylabel("Stärke [dB]")
plt.title("Sprache: geglättetes Spektrum vor und nach dem Hochpass")
plt.legend()
plt.grid(True)
plt.show()

> **Aufgabe 2.5:** Probieren Sie `grenzfrequenz_tp` = 300, 1000 und 4000 bei der Musik und `grenzfrequenz_hp` = 100, 500 und 2000 bei der Sprache. Tragen Sie in **Tabelle C** ein, wie der Klang sich verändert, ob die Sprache verständlich bleibt und ob sich der RMS-Wert ändert.

#### Zusatzaufgabe Z2.2: Frequenzgang und Ordnung (optional)

Wir schicken einen einzelnen Klick (**Impuls**) durch den Filter. Das Betragsspektrum der Antwort ist der **Frequenzgang**: 0 dB bedeutet "geht ungehindert durch". Weil `sosfiltfilt` zweimal filtert, ist das der quadrierte Betrag des einfachen Filters. Die x-Achse ist logarithmisch: 100 Hz, 1000 Hz und 10000 Hz haben gleiche Abstände.

In [ ]:
z_grenzfrequenz = 1000     # Hz
z_ordnung = 4              # Vorschläge: 2, 4, 8

impuls = np.zeros(8192)
impuls[4096] = 1.0                                                  # ein Klick in der Mitte
antwort_filter = tiefpass(impuls, sr_musik, z_grenzfrequenz, ordnung=z_ordnung)
gang_db = 20 * np.log10(np.abs(np.fft.rfft(antwort_filter)) + 1e-12)
frequenzen = np.fft.rfftfreq(len(impuls), d=1 / sr_musik)

plt.figure(figsize=(10, 3))
plt.semilogx(frequenzen[1:], gang_db[1:])
plt.axvline(z_grenzfrequenz, color="red", linestyle="--", label="Grenzfrequenz")
plt.axhline(-6, color="gray", linestyle=":", label="-6 dB")
plt.ylim(-80, 5)
plt.xlabel("Frequenz [Hz] (logarithmische Achse)")
plt.ylabel("Verstärkung [dB]")
plt.title("Frequenzgang des Tiefpasses (vorwärts und rückwärts)")
plt.legend()
plt.grid(True)
plt.show()

> **Z2.2:** Vergleichen Sie `z_ordnung` = 2, 4 und 8. Wie verändert sich die Steilheit? Bei welcher Dämpfung schneidet die Kurve die Grenzfrequenz?

## 5 Auswertung und Diskussion

**Tabelle A: Eigenschaften der Aufnahmen (Aufgabe 2.1)**

| Datei | Abtastrate [Hz] | Kanäle | Samples | Dauer [s] | Spitzenwert | Samples ab 0.999 |
|---|---|---|---|---|---|---|
| speech.wav | | | | | | |
| music.wav | | | | | | |
| sound.wav | | | | | | |

**Tabelle B: Wellenform und Höreindruck (Aufgabe 2.2)**

| Aufnahme | Wellenform (Pausen, gleichmäßig, lauter werdend, ...) | Höreindruck |
|---|---|---|
| Sprache | | |
| Musik | | |
| Rennwagen | | |

**Tabelle C: Filter (Aufgabe 2.5)**

| Signal | Filter | Grenzfrequenz [Hz] | Klang und Verständlichkeit | RMS gestiegen, gesunken, gleich? |
|---|---|---|---|---|
| Musik | Tiefpass | 300 | | |
| Musik | Tiefpass | 1000 | | |
| Musik | Tiefpass | 4000 | | |
| Sprache | Hochpass | 100 | | |
| Sprache | Hochpass | 500 | | |
| Sprache | Hochpass | 2000 | | |

**Fragen**
1. Warum reicht für Sprache oft eine Abtastrate von 16000 Hz, während Musik häufig mit 44100 Hz gespeichert wird?
2. Warum zeigt das Spektrum einer ganzen Sprachaufnahme breite Bereiche statt einzelner scharfer Spitzen wie beim Testton?
3. Was haben Sie in der Differenz (links - rechts) gehört? Wie erklären Sie Ihre Beobachtung?
4. Ab welcher Tiefpass-Grenzfrequenz klang die Musik für Sie noch natürlich? Begründen Sie mit Ihrem Höreindruck und dem Spektrum.
5. Wie haben sich Klang, Verständlichkeit und RMS-Wert der Sprache beim Hochpass mit 2000 Hz verändert? Welche Frequenzanteile wurden gedämpft?

**Ihre Antworten:**
1. 
2. 
3. 
4. 
5. 

---
## 6 Quellen und Audiodaten

Die Aufnahmen stammen von Dritten. Für dieses Labor wurden nur Ausschnitte herausgeschnitten und als WAV-Dateien gespeichert (20 ms Ein- und Ausblenden an den Schnittstellen, sonst keine Bearbeitung). Alle Details stehen in [`DATA_SOURCES.md`](https://github.com/Lamhour-Mohamed-Akram/audio-dsp-lab/blob/main/DATA_SOURCES.md).

- **`speech.wav`** (Sprache, 16000 Hz, mono, 15,9 s): LibriVox-Hörbuch "Fabeln" von Magnus Gottfried Lichtwer, Abschnitt "Das Wiesel und die Hühner", gelesen von **marham63**. Lizenz: **Public Domain**. Quelle: <https://archive.org/details/fabeln_1905_librivox>. Ausschnitt 12,8 s bis 28,7 s, umgetastet auf 16000 Hz.
- **`music.wav`** (Musik, 44100 Hz, stereo, 20,0 s): Julius Fučík, "Entrance of the Gladiators" ("Einzug der Gladiatoren"), gespielt von der **United States Marine Band**. Lizenz: **Public Domain** (Werk der US-Regierung). Quelle: <https://commons.wikimedia.org/wiki/File:Julius_Fu%C4%8D%C3%ADk%27s_%22Entrance_of_the_Gladiators%22,_performed_by_the_United_States_Marine_Band.flac>. Ausschnitt 24 s bis 44 s, als 16-Bit-WAV gespeichert.
- **`sound.wav`** (Rennwagen, 22050 Hz, stereo, 8,0 s): "Audi R8 (2000)", aufgenommen von **Edvvc** beim Goodwood Festival of Speed 2009. Lizenz: **CC BY-SA 3.0** (<https://creativecommons.org/licenses/by-sa/3.0/>). Quelle: <https://commons.wikimedia.org/wiki/File:Audi_R8_(2000).ogg>. Ausschnitt 4 s bis 12 s, umgetastet auf 22050 Hz. Der Ausschnitt steht ebenfalls unter CC BY-SA 3.0.